In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/live-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 03 · Tool-Calling Fine-Tuning — cut the cost (LIVE)

> **LIVE DEMO LAB.** Resources are suffixed `-live`. The fine-tune job is submitted but not
> awaited — pivot to the pre-demo lab to show the finished model while this one trains.

Every voice turn ships ~800–1,200 tokens of tool schemas before the member says a word, and the team pays for all of them. Tool-calling fine-tuning bakes the Acme tools into the model: train on multi-turn tool-calling traces, then call the same prompt three ways — full schemas, names-only, and no `tools` array at all — and compare the prompt-token cost. You get the same correct `verify_member_identity` call with ~80% fewer prompt tokens. *Same call, millions of tokens a month cheaper.*

---
## Step 1 — Config & client

In [ ]:
import os, json, time, requests
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

load_dotenv()

AZURE_OPENAI_ENDPOINT    = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_MODEL               = os.environ.get('BASE_MODEL', 'gpt-4o-mini-2024-07-18')
BASE_DEPLOYMENT          = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
SUBSCRIPTION_ID          = os.environ['AZURE_SUBSCRIPTION_ID']
RESOURCE_GROUP           = os.environ['AZURE_RESOURCE_GROUP']
RESOURCE_NAME            = os.environ['AZURE_RESOURCE_NAME']
TENANT_ID                = os.environ.get('AZURE_TENANT_ID')

TOOLS_SCHEMA_FILE  = Path('data/acme_tools_schema.json')
TRAINING_DATA_FILE = Path('data/acme_tool_calling_training_data.json')
TOOLS_JSONL        = Path('data/acme_tools.jsonl')

_cred = DefaultAzureCredential(interactive_browser_tenant_id=TENANT_ID) if TENANT_ID else DefaultAzureCredential()

client = AzureOpenAI(
    azure_endpoint          = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = lambda: _cred.get_token('https://cognitiveservices.azure.com/.default').token,
    api_version             = AZURE_OPENAI_API_VERSION,
)

ACME_TOOLS = json.loads(TOOLS_SCHEMA_FILE.read_text(encoding='utf-8'))
print(f'Loaded {len(ACME_TOOLS)} Acme tools:')
for t in ACME_TOOLS:
    print(f'  - {t["function"]["name"]}')

---
## Step 2 — Build the training JSONL

Each record carries `messages` (the full multi-turn conversation including
tool calls) **plus** the `tools` array. Assistant messages that have
`tool_calls` must NOT carry `content: null` — that breaks Azure validation,
so we strip it.

In [ ]:
training_examples = json.loads(TRAINING_DATA_FILE.read_text(encoding='utf-8'))

with open(TOOLS_JSONL, 'w', encoding='utf-8-sig') as f:
    for ex in training_examples:
        clean_msgs = []
        for m in ex['messages']:
            cm = dict(m)
            # Azure tool-calling SFT: assistant tool-call messages must not have content: null
            if cm.get('role') == 'assistant' and cm.get('tool_calls') and cm.get('content') is None:
                cm.pop('content', None)
            clean_msgs.append(cm)
        rec = { 'messages': clean_msgs, 'tools': ACME_TOOLS }
        f.write(json.dumps(rec) + '\n')

print(f'Wrote {len(training_examples)} examples to {TOOLS_JSONL}')
print('\nSample (first 600 chars):')
print(TOOLS_JSONL.read_text(encoding='utf-8-sig')[:600] + '...')

---
## Step 3 — Token budget BEFORE fine-tuning

Eyeball how many tokens we'd spend just sending the tool definitions every
single turn.

In [ ]:
full_tools_json     = json.dumps(ACME_TOOLS)
names_only          = json.dumps([{ 'type': 'function', 'function': { 'name': t['function']['name'], 'parameters': t['function']['parameters'] } } for t in ACME_TOOLS])

approx_full  = len(full_tools_json) // 4   # ~4 chars per token
approx_names = len(names_only) // 4

print(f'Approx tokens per turn (full schemas, with descriptions): ~{approx_full}')
print(f'Approx tokens per turn (names + params only)            : ~{approx_names}')
print(f'Approx tokens per turn (no tools array)                 : ~0')
print(f'\nSavings if we can drop the array entirely on a fine-tuned model: ~{approx_full} tokens / turn')

---
## Step 4 — BEFORE: base model with FULL tool schemas

The base model needs every parameter description; we're just checking it works.

In [ ]:
SYSTEM_PROMPT = (
    'You are a Acme Health Member Services voice assistant. Use the '
    'available tools to verify identity, look up prescriptions, request '
    'refills, find in-network providers, and calculate medication prices. '
    'Always verify identity before disclosing protected health information.'
)

DEMO_PROMPT = 'Hi this is Maria Rodriguez, DOB 7/12/1982, member MEM-099. Can you refill my metformin and ship it mail order?'

def call_with_tools(deployment, tools):
    return client.chat.completions.create(
        model       = deployment,
        messages    = [
            { 'role': 'system', 'content': SYSTEM_PROMPT },
            { 'role': 'user',   'content': DEMO_PROMPT },
        ],
        tools       = tools,
        temperature = 0.0,
        max_tokens  = 250,
    )

def show_response(label, resp):
    print(f'\n=== {label} ===')
    msg = resp.choices[0].message
    print(f'  prompt_tokens     : {resp.usage.prompt_tokens}')
    print(f'  completion_tokens : {resp.usage.completion_tokens}')
    if msg.tool_calls:
        for tc in msg.tool_calls:
            print(f'  tool_call         : {tc.function.name}({tc.function.arguments})')
    if msg.content:
        print(f'  content           : {msg.content}')

resp_base_full = call_with_tools(BASE_DEPLOYMENT, ACME_TOOLS)
show_response('BASE MODEL + FULL SCHEMAS', resp_base_full)

---
## Step 5 — Submit the tool-calling SFT job

In [ ]:
# Step 5 - Submit job. Idempotent: re-running this cell will NOT
# create a duplicate job - it checks .tools_job_id first.
import os, time
from pathlib import Path

print('--- Step 5 starting...', flush=True)

_job_marker = Path('.tools_job_id_live')
if _job_marker.exists():
    job_id = _job_marker.read_text().strip()
    s = client.fine_tuning.jobs.retrieve(job_id)
    print(f'  existing job found: {job_id}', flush=True)
    print(f'  status: {s.status}', flush=True)
    training_file_id = s.training_file
else:
    upload = client.files.create(file=open(TOOLS_JSONL, 'rb'), purpose='fine-tune')
    training_file_id = upload.id
    print(f'  uploaded: {training_file_id} ({upload.status})', flush=True)

    for _ in range(60):
        f = client.files.retrieve(training_file_id)
        if f.status == 'processed':
            break
        if f.status in ('error', 'deleted'):
            raise RuntimeError(f'File processing failed: {f.status}')
        time.sleep(5)

    job = client.fine_tuning.jobs.create(
        training_file = training_file_id,
        model         = BASE_MODEL,
        suffix        = 'acme-tools-live',
        seed          = 42,
        hyperparameters = { 'n_epochs': 3 },
        extra_body    = { 'trainingType': 'GlobalStandard' },
    )
    job_id = job.id
    _job_marker.write_text(job_id)
    print(f'  job: {job_id} ({job.status})', flush=True)

print('--- Step 5 done. ---', flush=True)


---
## Step 6 — Monitor

In [ ]:
# [LIVE DEMO] Non-blocking monitor.
# This cell does a SINGLE status check and then returns immediately so the
# presenter can pivot to the pre-demo lab while the job cooks server-side.
# Re-run this cell at the end of the demo to confirm completion.
import os, time
from dotenv import load_dotenv

if 'client' not in globals():
    load_dotenv()
    from openai import AzureOpenAI
    from azure.identity import DefaultAzureCredential, get_bearer_token_provider
    _cred = DefaultAzureCredential()
    _tp = get_bearer_token_provider(_cred, 'https://cognitiveservices.azure.com/.default')
    client = AzureOpenAI(
        azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
        api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
        azure_ad_token_provider=_tp,
    )

job_id = globals().get('job_id')
if not job_id:
    print('  no live job_id in scope. Run the previous (submit) step first.')
else:
    js = client.fine_tuning.jobs.retrieve(job_id)
    print(f'  job_id: {job_id}')
    print(f'  status: {js.status}')
    if js.status == 'succeeded':
        fine_tuned_model = js.fine_tuned_model
        print(f'  fine-tuned model: {fine_tuned_model}')
    elif js.status in ('failed', 'cancelled'):
        print(f'  job ended unexpectedly: {js.status}')
        print(getattr(js, 'error', None))
    else:
        print('  >>> [live demo] not blocking. Pivot to the pre-demo lab now.')
        print('  >>> Refresh Foundry portal -> Build -> Fine-tuning to show the run.')
        print('  >>> Re-run this cell at end of demo for final status.')


---
## Step 7 — Plot metrics

In [ ]:
final_job = client.fine_tuning.jobs.retrieve(job_id)
if final_job.result_files:
    content = client.files.content(final_job.result_files[0]).read()
    Path('data/acme_tools_results_live.csv').write_bytes(content)
    df = pd.read_csv('data/acme_tools_results_live.csv')

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.set_title('Acme Tool-Calling Fine-Tune', fontweight='bold')
    if 'train_loss' in df.columns:
        ax.plot(df['step'], df['train_loss'], label='Train Loss', linewidth=1.5)
    if 'train_mean_token_accuracy' in df.columns:
        ax2 = ax.twinx()
        ax2.plot(df['step'], df['train_mean_token_accuracy'], label='Train Acc', color='green', linewidth=1.5)
        ax2.set_ylabel('Token Accuracy', color='green')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.legend(loc='upper right'); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig('data/acme_tools_metrics_live.png', dpi=150); plt.show()

---
## Step 8 — Deploy

In [ ]:
# Step 8 - Deploy. Idempotent: short-circuits if deployment already Succeeded.
import os, time, json, requests
from pathlib import Path

print('--- Step 8 starting...', flush=True)

TOOLS_DEPLOYMENT_NAME = 'acme-tools-live-deployment'

fine_tuned_model = globals().get('fine_tuned_model') \
    or (Path('.tools_model_id_live').read_text().strip() if Path('.tools_model_id_live').exists() else None)
if not fine_tuned_model:
    raise RuntimeError('No fine-tuned model id. Run Step 6 first.')
print(f'  model: {fine_tuned_model}', flush=True)

auth = _cred.get_token('https://management.azure.com/.default').token
deploy_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}'
    f'/providers/Microsoft.CognitiveServices/accounts/{RESOURCE_NAME}'
    f'/deployments/{TOOLS_DEPLOYMENT_NAME}'
)
status_url = deploy_url + '?api-version=2024-10-01'

# Short-circuit if already deployed
g = requests.get(status_url, headers={'Authorization': f'Bearer {auth}'})
if g.status_code == 200 and g.json().get('properties', {}).get('provisioningState') == 'Succeeded':
    print('  -> already deployed, skipping', flush=True)
else:
    r = requests.put(
        deploy_url,
        params  = { 'api-version': '2024-10-01' },
        headers = { 'Authorization': f'Bearer {auth}', 'Content-Type': 'application/json' },
        json    = {
            'sku': { 'name': 'GlobalStandard', 'capacity': 1 },
            'properties': { 'model': { 'format': 'OpenAI', 'name': fine_tuned_model, 'version': '1' } },
        },
    )
    print(f'  PUT -> HTTP {r.status_code} {r.reason}', flush=True)
    if r.status_code not in (200, 201):
        print(json.dumps(r.json(), indent=2), flush=True)

    print('  waiting for Succeeded...', flush=True)
    t0 = time.time()
    while True:
        time.sleep(20)
        auth = _cred.get_token('https://management.azure.com/.default').token
        g = requests.get(status_url, headers={'Authorization': f'Bearer {auth}'})
        state = g.json().get('properties', {}).get('provisioningState', 'Unknown')
        elapsed = int(time.time() - t0)
        print(f'  [+{elapsed:>4}s] {state}', flush=True)
        if state == 'Succeeded': break
        if state in ('Failed', 'Canceled'):
            print(json.dumps(g.json(), indent=2), flush=True)
            break

print('--- Step 8 done. ---', flush=True)


---
## Step 9 — THE DEMO: 3 ways to call the same prompt

In [ ]:
# Strip descriptions: model still sees tool names + parameter shapes
names_and_shapes = []
for t in ACME_TOOLS:
    fn = t['function']
    params = json.loads(json.dumps(fn['parameters']))
    for prop in params.get('properties', {}).values():
        prop.pop('description', None)
    names_and_shapes.append({
        'type': 'function',
        'function': { 'name': fn['name'], 'parameters': params }
    })

def call_no_tools(deployment):
    return client.chat.completions.create(
        model       = deployment,
        messages    = [
            { 'role': 'system', 'content': SYSTEM_PROMPT },
            { 'role': 'user',   'content': DEMO_PROMPT },
        ],
        temperature = 0.0,
        max_tokens  = 250,
    )

resp_ft_names = call_with_tools(TOOLS_DEPLOYMENT_NAME, names_and_shapes)
show_response('FT MODEL + names/shapes only (no descriptions)', resp_ft_names)

resp_ft_zero  = call_no_tools(TOOLS_DEPLOYMENT_NAME)
show_response('FT MODEL + NO tools array at all', resp_ft_zero)

---
## Step 10 — Token-savings table

In [ ]:
rows = [
    ('Base + full schemas',           resp_base_full.usage.prompt_tokens),
    ('Fine-tuned + names/shapes only', resp_ft_names.usage.prompt_tokens),
    ('Fine-tuned + NO tools array',    resp_ft_zero.usage.prompt_tokens),
]
baseline = rows[0][1]
print(f"{'Configuration':<40} {'Prompt tokens':>15} {'Savings':>10}")
print('-' * 70)
for label, toks in rows:
    sav = f'{(1 - toks/baseline) * 100:.0f}%' if toks != baseline else '-'
    print(f'{label:<40} {toks:>15} {sav:>10}')

print(f'\nAt 1M calls/month, savings of ~{baseline - rows[2][1]} input tokens/call =')
print(f'~{(baseline - rows[2][1]) * 1_000_000 / 1_000_000:.0f}M tokens/month saved.')

---
## Step 11 — Cleanup

In [ ]:
try:
    client.files.delete(training_file_id); print(f'Deleted file: {training_file_id}')
except Exception as e:
    print(f'Could not delete file: {e}')

auth = _cred.get_token('https://management.azure.com/.default').token
r = requests.delete(deploy_url, params={ 'api-version': '2024-10-01' },
                    headers={ 'Authorization': f'Bearer {auth}' })
print(f'Deployment delete: HTTP {r.status_code}')

print(f'\nTool fine-tuned model name: {fine_tuned_model}')